In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_neutron_60K_278467_unopt_pbesol_magres.magres"


*This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file*

*Shiva Agarwal*

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'N'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted
Q = 0.0204 #electric quadrupole moment for 14N in barn

In [7]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

14N1 sigma:
 [[187.1111166    1.10282269  -4.26714268]
 [ -2.09184024 176.1892044   -3.59100854]
 [ -5.03225315  -4.52794332 192.58601823]]

14N2 sigma:
 [[187.1111166   -1.10282269   4.26714268]
 [  2.09184024 176.1892044   -3.59100854]
 [  5.03225315  -4.52794332 192.58601823]]

14N3 sigma:
 [[187.1111166    1.10282269   4.26714268]
 [ -2.09184024 176.1892044    3.59100854]
 [  5.03225315   4.52794332 192.58601823]]

14N4 sigma:
 [[187.1111166   -1.10282269  -4.26714268]
 [  2.09184024 176.1892044    3.59100854]
 [ -5.03225315   4.52794332 192.58601823]]



In [8]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

14N1 sigma:
 1.3356347547526095

14N2 sigma:
 1.3356347547526033

14N3 sigma:
 1.3356347547526122

14N4 sigma:
 1.3356347547526028



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.404 -0.286 -1.006]
 [-0.286 -0.429  0.294]
 [-1.006  0.294  0.025]]

CS Tensor:
 [[187.111   1.103  -4.267]
 [ -2.092 176.189  -3.591]
 [ -5.032  -4.528 192.586]]

CS isotropic Tensor:
 [[185.295   0.      0.   ]
 [  0.    185.295   0.   ]
 [  0.      0.    185.295]]

CS symmetric Tensor:
 [[187.111  -0.495  -4.65 ]
 [ -0.495 176.189  -4.059]
 [ -4.65   -4.059 192.586]]

CS antisymmetric Tensor:
 [[ 0.     1.597  0.383]
 [-1.597  0.     0.468]
 [-0.383 -0.468  0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 1.33302053 -0.81561281 -0.51740772] 

 Unsorted Eigenvectors:
 [[-0.74687368  0.60988676 -0.2649865 ]
 [ 0.22578732 -0.14222803 -0.96373818]
 [ 0.62545966  0.77962127  0.03147833]] 

Sorted Eigenvalues: 
 [-0.51740772 -0.81561281  1.33302053] 

Sorted Eigenvectors: 
 [[-0.2649865   0.60988676 -0.74687368]
 [-0.96373818 -0.14222803  0.22578732]
 [ 0.03147833  0.77962127  0.62545966]] 


For CS tensor
 Unsorted Eigenvalues:
 [195.80732392 185.05501037 175.02400494] 

 Unsorted Eigenvectors:
 [[ 0.45722216 -0.87854333 -0.13823714]
 [ 0.16914504  0.23850033 -0.95629888]
 [-0.87311961 -0.41385891 -0.25764889]] 

Sorted Eigenvalues: 
 [185.05501037 175.02400494 195.80732392] 

Sorted Eigenvectors: 
 [[-0.87854333 -0.13823714  0.45722216]
 [ 0.23850033 -0.95629888  0.16914504]
 [-0.41385891 -0.25764889 -0.87311961]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.5174077204549468 -0.8156128109393307 1.3330205313943067
CSA Tensor Components δyy, δxx, δzz: 
 185.05501036631392 175.0240049400089 195.80732392293604


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 14N1: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   1.33302  |
+--------------+------------+
| etaq         |   0.223706 |
+--------------+------------+
| iso_cs (ppm) | 185.295    |
+--------------+------------+
| csa (ppm)    |  10.5119   |
+--------------+------------+
| etas         |   0.954254 |
+--------------+------------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/N_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.60988676 -0.2649865  -0.74687368]
 [-0.14222803 -0.96373818  0.22578732]
 [ 0.77962127  0.03147833  0.62545966]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
2.3121433891897443 51.28406645309082 16.820587840663872 

Direction cosine csa: 

[[-0.13823714 -0.87854333  0.45722216]
 [-0.95629888  0.23850033  0.16914504]
 [-0.25764889 -0.41385891 -0.87311961]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
58.095569678351374 150.82320702242055 -20.30148027237688 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -58.74504754628252 chi: 148.14619724682365 xi: -36.194522696973266 

